# Merging adult and ontogeny datasets for all-age analysis
This notebook combines adult and infant data into a single all-age dataset, aligns time variables, and prepares the merged data for Stan modeling.

In [11]:
import numpy as np
import pandas as pd
import json

In [12]:
df_new = pd.read_csv('/home/apoorva/Desktop/MZB-new-analysis/data/New_data/new_data_MZB.csv')
df_ont = pd.read_csv('/home/apoorva/Desktop/MZB-new-analysis/data/New_data/ontogeny_counts.csv')

In [13]:


#Replace Age at BMT for 2nd row with 74

df_new.loc[1, 'Age at BMT'] = 74


print(df_new.head(100))

#Max value of Mz counts from adult data
max_Mz = df_new['total_MZ'].max()
print("Max Mz counts in adult data:", max_Mz)

                                    Unnamed: 0  Age at BMT  Age at S1K  \
0       20181129_Bcells_SPLEEN_14d_001_019.fcs          74          88   
1       20181129_Bcells_SPLEEN_14d_002_020.fcs          74          88   
2           20181204_2-5wks_SPLEEN_001_018.fcs          74          92   
3           20181204_2-5wks_SPLEEN_002_019.fcs          41          59   
4             20181207_3wks_SPLEEN_001_048.fcs          74          95   
5             20181207_3wks_SPLEEN_002_049.fcs          41          62   
6             20181213_4wks_SPLEEN_001_024.fcs          74         102   
7             20181213_4wks_SPLEEN_002_025.fcs          41          69   
8        20181220_5wks_SPLEEN_CD1d_001_065.fcs          74         109   
9        20181220_5wks_SPLEEN_CD1d_002_066.fcs          41          76   
10             20190312_SPLEEN_54d_004_020.fcs          41         119   
11  20190205_MV61_BuChi_12w_Spleen_001_028.fcs          41         123   
12  20190205_MV61_BuChi_12w_Spleen_002

In [14]:
# Remove one point from the df_new dataframe
# df_new = df_new.drop(df_new[df_new['total_T1_AA4.1+'] < 2*10**5].index)
# df_ont = df_ont.drop(df_ont[df_ont['T1 AA4.1+_total'] < 10**6].index)


# Change column name to match the other dataframe
df_new1 = df_new.rename(columns = {'Age at S1K':'age.at.S1K'})
df_ont1 = df_ont.rename(columns = {'T1 AA4.1+_total':'total_T1_AA4.1+', 'MZ_total':'total_MZ', 'frac_MZ_Ki67+' : 'frac_Ki67+_MZ'}) 

df_counts_Nfd = df_new1.filter(items=['age.at.S1K', 'total_T1_AA4.1+', 'total_MZ', 'frac_Ki67+_MZ'])
df_counts_ont = df_ont1.filter(items=['age.at.S1K', 'total_T1_AA4.1+', 'total_MZ', 'frac_Ki67+_MZ'])


# combine in one dataframe
total_counts = pd.concat([df_counts_Nfd, df_counts_ont])
total_counts = total_counts.sort_values(by='age.at.S1K')

total_counts = total_counts.reset_index(drop=True)

df_new = df_new.sort_values(by='Age at S1K')

In [15]:
# data time points
data_time = total_counts['age.at.S1K'].values

# Remove the first 3 data points from the data_time array
data_time = data_time[4:]

data_time_Nfd = df_new['Age at S1K'].values

# Create a dataframe with unique age at S1K values
dfunique = total_counts.drop_duplicates(subset='age.at.S1K', keep='first')
df_Nfdunique = df_new.drop_duplicates(subset='Age at S1K', keep='first')
dfunique = dfunique.sort_values(by='age.at.S1K')
dfunique = dfunique.reset_index(drop=True)

# unique time points in data
solve_time = dfunique['age.at.S1K'].values
solve_time = solve_time[1:]  # Remove the first data points
solve_time_Nfd = df_Nfdunique['Age at S1K'].values
solve_ageatbmt = df_Nfdunique['Age at BMT'].values
print(solve_ageatbmt)

# create index map of data time to solve time for all ages
time_index_map = np.zeros(len(data_time), dtype=int)
for i, t in enumerate(solve_time):
    time_index_map[data_time == t] = i+1

# create index map of data time to solve time for adult data
time_index_map_Nfd = np.zeros(len(data_time_Nfd), dtype=int)
for i, t in enumerate(solve_time_Nfd):
    time_index_map_Nfd[data_time_Nfd == t] = i+1
    
print(time_index_map)
print(time_index_map_Nfd)

time_pred_solve= np.array(np.arange(11.1, 732.1, 0.1))

#time sequence for predictions specific to agebins within the data
time_pred1_solve= np.array(np.arange(52, 750, 1))
# time_pred3_solve= np.array(np.arange(90, 750, 1))
time_pred2_solve= np.array(np.arange(88, 750, 1))
time_pred1_bmt = 41.0
# time_pred3_bmt= 89.0
time_pred2_bmt= 74.0

print(solve_ageatbmt)
print(solve_time_Nfd)
print(len(time_index_map_Nfd))


[ 41  41  41  41  74  74  74  74  74  41  41  42  51  65  58  65 101  89
  87]
[ 1  1  1  1  1  2  2  2  2  3  3  4  4  4  4  4  4  5  5  5  5  6  6  6
  6  7  7  7  7  7  7  8  8  9  9  9  9  9 10 11 11 12 12 13 13 13 13 14
 14 15 16 17 18 19 20 21 22 23 24 25 25 26 27 28 29 30 31 32 33 34 35 35
 36 37 38 39 40 40 41 41]
[ 1  2  3  4  5  5  6  7  8  9 10 11 12 13 14 15 16 17 18 18 19 19]
[ 41  41  41  41  74  74  74  74  74  41  41  42  51  65  58  65 101  89
  87]
[ 59  62  69  76  88  92  95 102 109 119 123 124 141 158 212 219 291 306
 731]
22


In [16]:
#Binning the data for Nfd dataframe on equally sized aged bins based on age at BMT

print(data_time)
print(solve_time)

# Craete an array of lenth solve_time with values 0 for df_ont and Age at BMT for df_new

# Create an array of length solve_time with values 0 for df_ont and Age at BMT for df_new
age_at_bmt = np.zeros(len(solve_time), dtype=float)
for i, t in enumerate(solve_time):
    if t in solve_time_Nfd:
        age_at_bmt[i] = solve_ageatbmt[np.where(solve_time_Nfd == t)[0][0]]
    else:
        age_at_bmt[i] = 0.0

print(age_at_bmt)
print(len(age_at_bmt))

[ 14  14  14  14  14  18  18  18  18  19  19  20  20  20  20  20  20  22
  22  22  22  23  23  23  23  24  24  24  24  24  24  25  25  26  26  26
  26  26  27  28  28  29  29  31  31  31  31  33  33  36  39  43  53  57
  59  62  69  71  76  88  88  92  95 102 109 113 119 123 124 141 152 152
 158 212 219 291 306 306 731 731]
[ 14  18  19  20  22  23  24  25  26  27  28  29  31  33  36  39  43  53
  57  59  62  69  71  76  88  92  95 102 109 113 119 123 124 141 152 158
 212 219 291 306 731]
[  0.   0.   0.   0.   0.   0.   0.   0.   0.   0.   0.   0.   0.   0.
   0.   0.   0.   0.   0.  41.  41.  41.   0.  41.  74.  74.  74.  74.
  74.   0.  41.  41.  42.  51.   0.  65.  58.  65. 101.  89.  87.]
41


In [17]:
# Creating a list to use as an input in Stan model

data={"numobs" : len(data_time),  # Number of obervation throughout the age range
      "numobsNfd" : len(df_new['Nfd_MZ']),  # Number of obervation throughout the age range
      "numsolve" : len(solve_time),  # Number of unique time points in the data
      "numsolveNfd" : len(solve_time_Nfd),  # Number of unique time points in the data
      "data_time" : data_time,
      "time_index_map" : time_index_map,
      "time_index_map_Nfd" : time_index_map_Nfd,
      "solve_time" : solve_time,
      "solve_ageatBMT" : solve_ageatbmt, # Age at BMT for the solve time points
      "solve_time_Nfd" : solve_time_Nfd,
      "totalMZ" : np.array(total_counts['total_MZ'][4:]),
      "Nfd" : np.array(df_new['Nfd_MZ']),
      "dayspbmt" : np.array(df_new['Days postBMT']),
      # "ageatbmt" : np.array(age_at_bmt),
      "ki_donor" : np.array(df_new['frac_Ki67+_MZ_donor']),
      "ki_host" : np.array(df_new['frac_Ki67+_MZ_host']),
      "time_pred_solve" : time_pred_solve,
      "time_pred1_solve" : time_pred1_solve,
      "time_pred2_solve" : time_pred2_solve,
    #   "time_pred3_solve" : time_pred3_solve,
      "time_pred1_bmt" : time_pred1_bmt,
      "time_pred2_bmt" : time_pred2_bmt,
    #   "time_pred3_bmt" : time_pred3_bmt,
      "numpred" : len(time_pred_solve),
      "numpred1" : len(time_pred1_solve),
      "numpred2" : len(time_pred2_solve),
    #   "numpred3" : len(time_pred3_solve)
      }

# Export data to JSON
def numpy_array_encoder(obj):
    if isinstance(obj, np.ndarray):
        return obj.tolist()  # Convert numpy array to a list
    raise TypeError("Object of type {} is not JSON serializable".format(type(obj)))

# Serialize dictionary to JSON using custom encoder function
json_string = json.dumps(data, default=numpy_array_encoder)

file_path= "/home/apoorva/Desktop/MZB-new-analysis/data/newdata_mz_allage_2agebins.json"
with open(file_path, "w") as json_file:  json_file.write(json_string)

# print("Data saved to", file_path)